# FlashVSR-Pro — persistent Colab workflow

Use **Ubuntu 24.04 noble, native Python 3.13, PyTorch 2.11.0+cu128, CUDA Toolkit 12.8**, with an **A100 or L4 GPU**. T4/P100 and older Colab runtimes are unsupported. The CUDA version displayed by `nvidia-smi` describes the driver; setup checks the actual `nvcc` toolkit separately.

Run cells in order. Models and verified compiled wheels survive restarts in Google Drive. Computation happens on local `/content` storage, and every completed video segment is verified and saved to Drive. Only one notebook should write to the chosen Drive directory at a time.

The default **20 GB project budget includes a 2 GB reserve**. Lossless segments can exceed this budget on long/high-resolution videos; the job stops and keeps saved progress rather than deleting older results. Drive's account quota can differ from the mount's reported free space.


## 1. Paths, runtime and code revision

The default Drive root is `MyDrive/FlashVSR-Pro`. An existing model/wheel cache is reused after verification. Use the exact `code_commit` shown by the job list when resuming an older job. To test this unmerged PR, set `project_ref` to `codex/colab-workflow`; after merging, use `main` for new jobs.


In [ ]:
#@title Storage and model settings
from pathlib import Path
import json
import shutil
import subprocess
import sys
from google.colab import drive

project_ref = "main" #@param {type:"string"}
drive_directory = "/content/drive/MyDrive/FlashVSR-Pro" #@param {type:"string"}
drive_budget_gb = 20.0 #@param {type:"number"}
reserve_gb = 2.0 #@param {type:"number"}
mode = "tiny" #@param ["tiny", "tiny-long", "full"]
project = Path('/content/FlashVSR-Pro')
local_root = Path('/content/flashvsr')

if sys.version_info[:2] != (3, 13):
    raise RuntimeError('Select the current Colab Python 3.13 runtime.')
drive.mount('/content/drive')
drive_root = Path(drive_directory)

# Colab normally includes these. Install missing system tools only; do not
# install Ubuntu python3-dev (its interpreter may differ from Colab Python).
packages = {'git': 'git', 'g++': 'build-essential', 'ffmpeg': 'ffmpeg', 'ffprobe': 'ffmpeg'}
missing = sorted({package for tool, package in packages.items() if not shutil.which(tool)})
if missing:
    subprocess.run(['apt-get', 'update'], check=True)
    subprocess.run(['apt-get', 'install', '-y', *missing], check=True)
if not project.exists():
    subprocess.run(['git', 'init', str(project)], check=True)
    subprocess.run(['git', 'remote', 'add', 'origin', 'https://github.com/delvilo/FlashVSR-Pro.git'], cwd=project, check=True)
if subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=project, text=True).strip():
    raise RuntimeError('The checkout has local edits. Preserve them before selecting another revision.')
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', project_ref], cwd=project, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=project, check=True)
if not (project / 'flashvsr/colab/__main__.py').is_file():
    raise RuntimeError('Select a code revision containing this Colab workflow.')

def colab_command(action):
    return [sys.executable, '-m', 'flashvsr.colab', action, '--project', str(project),
            '--drive-root', str(drive_root), '--local-root', str(local_root),
            '--drive-budget-gb', str(drive_budget_gb), '--reserve-gb', str(reserve_gb)]

print('Code commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=project, text=True).strip())


## 2. Diagnose the environment

Checks OS, native Python headers, PyTorch, GPU/VRAM, Toolkit, compiler, FFmpeg, actual HEVC NVENC encoding, local space and managed Drive usage. A100 may select software encoding; inference still uses the GPU. Reports are saved in `diagnostics/` on Drive. Missing/incompatible core components stop setup before pip changes the environment.


In [ ]:
subprocess.run(colab_command('diagnose'), cwd=project, check=True)


## 3. Prepare dependencies, compiled kernels and models

Reuses Colab's matching PyTorch installation. A compatible wheel is restored from Drive and tested with a real sparse CUDA operation; a missing/corrupt/incompatible wheel is rebuilt locally. Only the selected mode's weights are downloaded. Cache hits still undergo integrity checks.

Each stage uses a fresh Python subprocess, so notebook imports cannot retain old NumPy/package versions. No virtual environment or kernel restart is required. Build details are retained in `diagnostics/setup.log`.


In [ ]:
subprocess.run(colab_command('setup') + ['--mode', mode], cwd=project, check=True)


## 4. Inspect saved jobs and storage

For resume, copy the job name and restore its settings (including `mode`, scale, segment size and code commit). Python/package version changes are rejected for an existing job. Select the saved revision in cell 1, or create a new job. Completed jobs verify their saved results and return without loading models.


In [ ]:
subprocess.run(colab_command('status'), cwd=project, check=True)


## 5. Select input and run

Place input videos under the Drive root's `inputs/` folder to avoid a second persistent copy. Inputs uploaded to `/content`, or outside that root, are copied into the new job's Drive folder before processing. Resume can leave the input field empty and use the saved source.

The default uses bounded segments and spatial tiling, preserves all audio tracks and saves each completed segment before starting another. A new job name creates a separate result directory. On interruption, set **resume_job = True**, retain the job name and all settings, then run again. Across VM resets, re-run setup and select the same job.


In [ ]:
#@title Video settings
from datetime import datetime, timezone
import uuid

input_path = "/content/drive/MyDrive/FlashVSR-Pro/inputs/input.mp4" #@param {type:"string"}
job_name = "" #@param {type:"string"}
resume_job = False #@param {type:"boolean"}
scale = 2.0 #@param {type:"number"}
segment_frames = 129 #@param {type:"integer"}
tile_size = 256 #@param {type:"integer"}
overlap = 24 #@param {type:"integer"}
seed = 0 #@param {type:"integer"}
color_fix = False #@param {type:"boolean"}
keep_audio = True #@param {type:"boolean"}
keep_segments_after_success = False #@param {type:"boolean"}

if resume_job and not job_name:
    raise ValueError('Choose an existing job name from the job list.')
if not job_name:
    job_name = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-') + uuid.uuid4().hex[:8]
print('Job name (keep this for resume):', job_name)
command = colab_command('run') + ['--job', job_name, '--mode', mode, '--scale', str(scale),
    '--segment-frames', str(segment_frames), '--tile-size', str(tile_size),
    '--overlap', str(overlap), '--seed', str(seed)]
if not resume_job:
    command += ['--input', input_path]
else:
    command.append('--resume')
if color_fix:
    command.append('--color-fix')
if not keep_audio:
    command.append('--no-audio')
if keep_segments_after_success:
    command.append('--keep-temp')
subprocess.run(command, cwd=project, check=True)


## 6. Results are already saved to Drive

Each `results/JOB/` contains `enhanced.mp4`, `report.json`, `run.jsonl`, `parameters.json` and `environment.json`. Completion is recorded only after every saved file passes read-back verification. Intermediate segments are removed after success unless requested above. Failed jobs retain saved segments; existing results and model/wheel caches are never automatically evicted.

For a browser download, enable the optional cell below. Large videos are best retrieved directly from Drive. Delete old results/jobs manually when no longer needed; Drive Trash may also consume quota.


In [ ]:
#@title Optional browser download
browser_download = False #@param {type:"boolean"}
completion_path = drive_root / 'jobs' / job_name / 'completion.json'
if not completion_path.is_file():
    raise RuntimeError('No verified completed result; inspect the job status and resume.')
completion = json.loads(completion_path.read_text())
result_video = drive_root / completion['video']
print('Saved video:', result_video)
print('Reports:', result_video.parent)
if browser_download:
    from google.colab import files
    files.download(str(result_video))


## 7. Optional GPU acceptance

This downloads the remaining mode weights, then validates all three modes, segmented inference and completed-job resume. It records resolution, frames, audio and VRAM. Reports are saved to Drive for review; this is separate from the short kernel test during setup.


In [ ]:
#@title Run all-mode GPU validation
run_gpu_acceptance = False #@param {type:"boolean"}
if run_gpu_acceptance:
    subprocess.run(colab_command('setup') + ['--mode', 'all'], cwd=project, check=True)
    subprocess.run(colab_command('validate'), cwd=project, check=True)
